<a href="https://colab.research.google.com/github/Mmbsaksd/transformers/blob/main/BPE_Explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Byte Pair Encoding (BPE) — Explained with Two English Examples

**Byte Pair Encoding (BPE)** is a tokenization algorithm used by many modern language models (like GPT and others) to break words into smaller, reusable pieces called **subword tokens**.

### Why not just split on whole words?
- A vocabulary of *all* whole words would be huge, and the model would still fail on new/unseen words (like `"unhappiness"` if it only ever saw `"happy"`).

### Why not just split into single characters?
- That gives a tiny vocabulary, but sequences become very long and the model loses the notion of common word chunks (like `"ing"`, `"est"`, `"un"`).

**BPE is the middle ground.** It starts with individual characters and repeatedly merges the *most frequently occurring pair* of symbols into a single new symbol — building up common subwords step by step, purely by counting statistics in a text corpus.

In this notebook, we will:
1. Build BPE **from scratch in plain Python** (no libraries needed) so every step is visible.
2. Run it on **Example 1**: a tiny toy corpus, and watch the merge rules get learned one by one.
3. Use those learned merge rules on **Example 2**: brand-new words the algorithm has never seen, to see how BPE tokenizes them.

> This notebook is self-contained and will run top-to-bottom in Google Colab with no extra installs.

## Step 0: The BPE Algorithm in Plain English

1. Split every word in the training corpus into individual **characters**, and mark the end of each word with a special symbol `</w>` (so the model can tell `"low"` apart from `"lower"`).
2. Count how often every **adjacent pair of symbols** occurs across the whole corpus (weighted by word frequency).
3. Find the **single most frequent pair** and merge it everywhere into one new symbol.
4. Repeat steps 2–3 for a fixed number of merges (or until no pairs are left).
5. The list of merges you learned, in order, **is your BPE tokenizer**. To tokenize a brand-new word later, you just apply the same merges, in the same order, to that new word's characters.

In [1]:
# --- Step 1: Define our training corpus ---
# Each key is a word, each value is how many times it occurred in our "text".
# (In a real system this would be counted from millions of words of text.)
corpus = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

# --- Step 2: Break every word into characters, plus an end-of-word marker ---
# The "</w>" marker matters: without it, BPE couldn't tell where one word
# ends and another begins once symbols start getting merged together.
def word_to_symbols(word):
    return list(word) + ["</w>"]

# vocab maps: tuple-of-symbols -> frequency
# e.g. ('l', 'o', 'w', '</w>') -> 5
vocab = {tuple(word_to_symbols(word)): freq for word, freq in corpus.items()}


print("Starting vocabulary (every word split into individual characters):")
for word_symbols, freq in vocab.items():
    print(f"  {word_symbols}   (frequency={freq})")

Starting vocabulary (every word split into individual characters):
  ('l', 'o', 'w', '</w>')   (frequency=5)
  ('l', 'o', 'w', 'e', 'r', '</w>')   (frequency=2)
  ('n', 'e', 'w', 'e', 's', 't', '</w>')   (frequency=6)
  ('w', 'i', 'd', 'e', 's', 't', '</w>')   (frequency=3)


In [2]:
# --- Step 3: Helper functions for BPE ---
from collections import defaultdict

def get_pair_frequencies(vocab):
    """
    Look at every word in the vocabulary and count how many times each
    ADJACENT pair of symbols occurs, weighted by that word's frequency.

    Example: if ('n','e','w','e','s','t','</w>') has frequency 6,
    then the pair ('e','w') gets +6, the pair ('w','e') gets +6, etc.
    """
    pair_counts = defaultdict(int)
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_counts[pair] += freq
    return pair_counts




def merge_pair_in_vocab(pair_to_merge, vocab_in):
    """
    Go through every word in the vocabulary and merge every occurrence
    of `pair_to_merge` into one new combined symbol.

    Example: merging ('e', 's') turns ('n','e','w','e','s','t','</w>')
    into ('n','e','w','es','t','</w>').
    """
    merged_symbol = "".join(pair_to_merge)
    vocab_out = {}
    for symbols, freq in vocab_in.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            # If the current position matches the pair we're merging, combine them
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair_to_merge:
                new_symbols.append(merged_symbol)
                i += 2  # skip both symbols we just merged
            else:
                new_symbols.append(symbols[i])
                i += 1
        vocab_out[tuple(new_symbols)] = freq
    return vocab_out

print("Helper functions defined: get_pair_frequencies() and merge_pair_in_vocab()")

Helper functions defined: get_pair_frequencies() and merge_pair_in_vocab()


In [3]:
num_merges = 8
merge_history = []
current_vocab = dict(vocab)


for step in range(1, num_merges + 1):
    pair_counts = get_pair_frequencies(current_vocab)
    if not pair_counts:
        print("No more pairs left to merge — stopping early.")
        break

    # Pick the single most frequent adjacent pair across the whole corpus
    best_pair = max(pair_counts, key=pair_counts.get)
    best_pair_freq = pair_counts[best_pair]

    # Apply that merge everywhere in the vocabulary
    current_vocab = merge_pair_in_vocab(best_pair, current_vocab)
    merge_history.append(best_pair)

    print(f"Merge #{step}: {best_pair[0]!r} + {best_pair[1]!r}  ->  "
          f"{''.join(best_pair)!r}   (seen together {best_pair_freq} times)")

print("\nFinal vocabulary after all merges:")
for symbols, freq in current_vocab.items():
    print(f"  {symbols}   (frequency={freq})")

Merge #1: 'e' + 's'  ->  'es'   (seen together 9 times)
Merge #2: 'es' + 't'  ->  'est'   (seen together 9 times)
Merge #3: 'est' + '</w>'  ->  'est</w>'   (seen together 9 times)
Merge #4: 'l' + 'o'  ->  'lo'   (seen together 7 times)
Merge #5: 'lo' + 'w'  ->  'low'   (seen together 7 times)
Merge #6: 'n' + 'e'  ->  'ne'   (seen together 6 times)
Merge #7: 'ne' + 'w'  ->  'new'   (seen together 6 times)
Merge #8: 'new' + 'est</w>'  ->  'newest</w>'   (seen together 6 times)

Final vocabulary after all merges:
  ('low', '</w>')   (frequency=5)
  ('low', 'e', 'r', '</w>')   (frequency=2)
  ('newest</w>',)   (frequency=6)
  ('w', 'i', 'd', 'est</w>')   (frequency=3)
